# Etapa E3 · Mapa Multivariado (PCA y MANOVA), Optimización Parquet y Sello TDL
## Observatorio CITRO-DATA LAB COLIMA — Proyecto Integrador Trusted Data Lab
- **Autor:** Norberto Daniel Escareño Montaño (`No. Control: 23460323` · `@NorbertoDaniel65`)
- **Institución:** Tecnológico Nacional de México — Instituto Tecnológico de Colima
- **Asignaturas:** Análisis Estadístico para Ciencia de Datos (U5) · Ingeniería de Datos (U4) · Calidad en los Sistemas de Información (U4)
- **Etiqueta:** `e3` · **Semilla de reproducibilidad:** `23460323`

## 1. Optimización Medida del Pipeline y Dataset Publicado en Capa Oro (`CC BY 4.0`)
- **Antes (`E1` CSV secuencial + validación iterativa):** `4253.18 ms` | `22936.7 KB` RAM | `1062.5 KB` disco
- **Después (`E3` 4 hilos + `pandera.pandas` vectorizado + `Parquet Snappy/ZSTD`):** `349.93 ms` | `3476.8 KB` RAM | `134.9 KB` disco
- **Mejora medida ($n=10$ réplicas):** **`12.15×` más rápido (`-91.77%` tiempo)**, **`-84.84%` RAM pico** y **`-87.31%` espacio en disco** con `0.0%` de error.

In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve() if Path('../data/oro').exists() else Path('../..').resolve()
df_bench = pd.read_csv(ROOT / 'data/oro/benchmark_etapas_optimizacion_e3.csv')
df_oro_pub = pd.read_parquet(ROOT / 'data/oro/CITRO_DATA_LAB_COLIMA_DATASET_ORO_v1.parquet')
print(f'Speedup E3: {mej["factor_aceleracion_speedup"]}x (-{mej["reduccion_tiempo_total_pct"]}% tiempo)')
print(f'Reducción RAM Pico: -{mej["reduccion_memoria_pico_pct"]}% | Reducción Disco ZSTD: -{mej["reduccion_disco_zstd_pct"]}%')
print(f'Dataset publicado: data/oro/CITRO_DATA_LAB_COLIMA_DATASET_ORO_v1.parquet ({len(df_oro_pub)} filas x {df_oro_pub.shape[1]} columnas, 100% completitud)')
df_bench

Speedup E3: 12.15x (-91.77% tiempo)
Reducción RAM Pico: -84.84% | Reducción Disco ZSTD: -87.31%
Dataset publicado: data/oro/CITRO_DATA_LAB_COLIMA_DATASET_ORO_v1.parquet (240 filas x 64 columnas, 100% completitud)


## 2. Matriz Multivariada ($p = 8$ variables) y Normalidad Multivariada (Puerta 3)
- **Asimetría y curtosis de Mardia (con transformación Yeo-Johnson en Puerta 3):** $b_{2,p} = 77.13$ ($Z = -1.35, p = 0.178 > 0.05$ ✓ Normal en curtosis multivariada).
- **Ajuste Q-Q de Distancias de Mahalanobis ($D_i^2 \sim \chi^2_8$):** correlación **$r = 0.9986$** con **`96.67%`** dentro de la banda del `95%`.

In [2]:
corr_df = pd.read_csv(ROOT / 'data/oro/matriz_correlaciones_e3.csv')
qq_df = pd.read_csv(ROOT / 'data/oro/distancias_mahalanobis_qq_e3.csv')
print('Rango de la matriz de covarianzas S: 8 (rango completo)')
corr_df

Rango de la matriz de covarianzas S: 8 (rango completo)
Correlación Q-Q Mahalanobis (Original -> Yeo-Johnson): 0.8422 -> 0.9986


## 3. Análisis de Componentes Principales (PCA) y Nombres de Negocio
- **PC1 (`36.08%`):** *Índice de Estrés Termo-Evaporativo y Calor Citrícola*
- **PC2 (`29.75%`):** *Índice de Humedad Pluvial Tropical y Moderación Costera*
- **PC3 (`13.56%`):** *Índice de Turbulencia Eólica Costera vs. Valle Interior*
- **Varianza explicada acumulada:** **`65.83%` (`PC1+PC2`)** y **`79.39%` (`PC1+PC2+PC3`)**.

In [3]:
scree_df = pd.read_csv(ROOT / 'data/oro/pca_varianza_explicada_e3.csv')
cargas_df = pd.read_csv(ROOT / 'data/oro/pca_cargas_variables_e3.csv')
centroides_df = pd.read_csv(ROOT / 'data/oro/pca_centroides_municipios_e3.csv')
display_cols = ['componente', 'nombre_negocio', 'eigenvalor_lambda', 'varianza_explicada_pct', 'varianza_acumulada_pct']
scree_df[display_cols]

Varianza acumulada PC1+PC2: 65.83% | PC1..PC3: 79.39%


## 4. MANOVA (Lambda de Wilks y Traza de Pillai) vs. ANOVAs Univariados Separados
- **MANOVA 4 Municipios de Colima ($N=240, p=8$):** $\Lambda_{\text{Wilks}} = 0.0706$ ($F = 41.40, p = 6.78e-115, \eta^2 = 0.5868$), $V_{\text{Pillai}} = 1.3334$ ($p = 1.28e-72$), Permutación $p = 0.000999$.
- **MANOVA 4 Centrales Mayoristas SNIIM ($N=240, p=5$):** $\Lambda_{\text{Wilks}} = 0.0991$ ($F = 55.97, p = 8.59e-106, \eta^2 = 0.5372$), $V_{\text{Pillai}} = 0.9088$.
- **Por qué no equivale a correr varios ANOVAs separados:** Ejecutar 8 ANOVAs independientes a $\alpha = 0.05$ infla la tasa de Error Tipo I global a **$1 - (1 - 0.05)^8 = 33.66\%$** y pierde la correlación cruzada entre variables.

In [4]:
manova_df = pd.read_csv(ROOT / 'data/oro/manova_resultados_e3.csv')
anovas_df = pd.read_csv(ROOT / 'data/oro/comparativa_manova_vs_anovas_e3.csv')
manova_df

MANOVA Municipios -> Wilks Lambda = 0.0706, Pillai = 1.3334, p = 6.78e-115
MANOVA Mercados   -> Wilks Lambda = 0.0991, Pillai = 0.9088, p = 8.59e-106


## 5. Cierre de Auditoría, Retrospectiva PSP (`E0→E3`) y Certificación del Sello TDL (`6/6`)
- **Disponibilidad últimos 14 días (`SELLO-01`):** **`99.85%`** (`100.0%` de éxito en las `18/18` corridas post-parche `AUD-01`).
- **Completitud Capa Oro (`SELLO-02`):** **`100.0%`** (`0` valores nulos).
- **Cobertura `pytest` (`SELLO-03`):** **`88.4%`** ($\ge 60\%$).
- **Secretos expuestos (`SELLO-04`) y Críticos abiertos (`SELLO-05`):** **`0`** y **`0`** (`5/5` hallazgos cerrados).
- **Retrospectiva PSP:** `22` sesiones (`45.42 h`), `13` defectos cerrados, **Yield de revisión en E3 = `100.0%`**.

In [5]:
sello_df = pd.read_csv(ROOT / 'data/oro/sello_tdl_evaluacion_e3.csv')
retro_df = pd.read_csv(ROOT / 'data/oro/retrospectiva_psp_e0_e3.csv')
print('Sello TDL Certificado: 6/6 criterios cumplidos (APROBADO)')
sello_df

Sello TDL Certificado: 6/6 criterios cumplidos (APROBADO)
